# 02. Três métodos de seleção e votação das features

Os três métodos rodam aqui, no mesmo notebook, e todos enxergam só o treino. O pré-processamento (imputação e padronização) também é ajustado só no treino. Como não usamos validação cruzada, não existe seleção dentro de fold: a validação e o teste simplesmente não entram nesta etapa.

- **SelectKBest com `f_regression`:** teste F de cada atributo contra o alvo, um de cada vez. Mede relação linear e é a estatística adequada para alvo contínuo.
- **Importância do Random Forest:** quanto cada atributo reduz o erro nas divisões das árvores. Pega relações não lineares e interações.
- **RFE com Ridge:** tira, um por vez, o atributo de menor coeficiente até sobrarem k. Olha os atributos em conjunto, então descarta os redundantes.

**Regra de votação, fixada no `config.yaml` antes de rodar:** cada método vota nas suas k = 10 melhores. É eleita a feature com pelo menos 2 dos 3 votos. Se a maioria eleger menos de 6, o conjunto é completado pelo menor ranking médio dos três métodos, que também desempata a ordem. Esse conjunto é o único usado pelos dois modelos do ciclo.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import ciclo, features, preprocessamento

cfg = ciclo.carregar_config()
estado = ciclo.atual()
mlflow = ciclo.configurar_mlflow(cfg)
dataset, janelas = preprocessamento.carregar(cfg)
pd.set_option("display.precision", 3)
print(f"ciclo {estado['cycle_id']} | {estado['data_version']}")

ciclo 20261109-184535 | dv-20261102-97fa6541


In [2]:
treino = janelas["treino"]
candidatas = [c for c in dataset.columns if c != "alvo"]
print(f"{len(candidatas)} candidatas, treino de {treino.index.min():%d/%m/%Y} a {treino.index.max():%d/%m/%Y} ({len(treino)} dias)")

ranking = features.tres_metodos(treino, candidatas, cfg, cfg["projeto"]["semente"])
ranking[["f_score", "rank_kbest", "rf_importancia", "rank_rf", "rfe_ranking", "rank_rfe"]].sort_values("rank_rf")

26 candidatas, treino de 07/01/2010 a 02/10/2025 (5748 dias)


,f_score,rank_kbest,rf_importancia,rank_rf,rfe_ranking,rank_rfe
tmed,6749.875,1.0,0.479,1.0,1,1.0
tmax,5747.710,2.0,0.131,2.0,10,19.0
vento_dir_cos,901.292,13.0,0.064,3.0,1,2.0
dia_ano_cos,2716.843,7.0,0.035,4.0,1,3.0
rajada,10.233,25.0,0.024,5.0,1,6.0
tmin,4794.029,4.0,0.021,6.0,1,8.0
vento_dir_sin,127.521,19.0,0.021,7.0,12,21.0
pressao_delta,665.917,14.0,0.021,8.0,1,9.0
chuva,39.473,20.0,0.020,9.0,4,13.0
dia_ano_sin,286.975,16.0,0.017,10.0,15,24.0


## Tabela de votos

In [3]:
votos, eleitas = features.votar(ranking, cfg)
print(f"eleitas ({len(eleitas)}):", ", ".join(eleitas))
votos[["rank_kbest", "rank_rf", "rank_rfe", "voto_kbest", "voto_rf", "voto_rfe", "votos", "ranking_medio", "eleita"]]

eleitas (8): tmed, dia_ano_cos, tmin, vento_dir_cos, tmax, radiacao, pressao_delta, rajada


,rank_kbest,rank_rf,rank_rfe,voto_kbest,voto_rf,voto_rfe,votos,ranking_medio,eleita
tmed,1.0,1.0,1.0,True,True,True,3,1.000,True
dia_ano_cos,7.0,4.0,3.0,True,True,True,3,4.667,True
tmin,4.0,6.0,8.0,True,True,True,3,6.000,True
vento_dir_cos,13.0,3.0,2.0,False,True,True,2,6.000,True
tmax,2.0,2.0,19.0,True,True,False,2,7.667,True
radiacao,8.0,11.0,4.0,True,False,True,2,7.667,True
pressao_delta,14.0,8.0,9.0,False,True,True,2,10.333,True
rajada,25.0,5.0,6.0,False,True,True,2,12.000,True
orvalho,3.0,13.0,11.0,True,False,False,1,9.000,False
amplitude,17.0,14.0,7.0,False,False,True,1,12.667,False


## Registro no MLflow

Um run de votação com os parâmetros da regra, as features eleitas, a tabela de votos completa (com rankings e importâncias) e as tags `cycle_id` e `data_version` do ciclo.

In [4]:
v = cfg["votacao"]
arquivo_votos = ciclo.pasta(estado) / "votos.csv"
votos.to_csv(arquivo_votos, index_label="feature")
with mlflow.start_run(run_name=f"{estado['cycle_id']}-votacao", tags=ciclo.tags(estado, "votacao")) as run:
    mlflow.log_params({"metodos": "selectkbest_f_regression,random_forest,rfe_ridge", "metodo_votacao": v["metodo"],
                       "k": v["k"], "votos_minimos": v["votos_minimos"], "min_features": v["min_features"],
                       "desempate": v["desempate"], "rf_arvores": v["rf_arvores"], "semente": cfg["projeto"]["semente"],
                       "features_eleitas": ",".join(eleitas)})
    mlflow.log_metric("n_eleitas", len(eleitas))
    mlflow.log_artifact(arquivo_votos)

estado["votacao"] = {"run_id": run.info.run_id, "features": eleitas, "n": len(eleitas)}
ciclo.salvar(estado)
print("run de votação:", run.info.run_id)

run de votação: 3763a59e244f47478971606ebf29fe5d
